# Reproducing Shaer on a Colab A100 (experiments E0 - E3)

Branch `dev` of the fork (`thebazshah/Shaer`), forked from the official repository at commit `bba73cb`.
Everything this notebook produces is written to **your Google Drive** (`DRIVE_ROOT` below).

| Step | What | Needs | Roughly |
|---|---|---|---|
| E0 | Re-score the authors' released poems with the repo's evaluator | any runtime (CPU is fine) | minutes |
| E1 | Generate the 3,481 test poems with the released adapter, score them | GPU, 24 GB+ | hours |
| E2 | Same for the untuned Yehia-7B | GPU, 24 GB+ | hours |
| E3 | Re-train Shaer (QLoRA, 4,236 steps), then generate and score | A100 40/80 GB class | the authors' run took 12.5 h |

**Runtime:** `Runtime > Change runtime type > A100 GPU`. The training stack is installed into its own Python 3.12
virtual environment with the repository's pinned versions (transformers 4.43.0, peft 0.12.0, bitsandbytes 0.43.3,
torch 2.3.1), so Colab's own preinstalled PyTorch / transformers versions do not matter.

**Before you start:** create a Hugging Face **read** token and add it as a Colab secret named `HF_TOKEN` (key icon in the
left sidebar, "Notebook access" on). The paper's base model
[`Navid-AI/Yehia-7B-preview`](https://huggingface.co/Navid-AI/Yehia-7B-preview) is gated; request access with that account.
**If access is not granted**, set `BASE_MODEL = "ALLaM-AI/ALLaM-7B-Instruct-preview"` in the first cell: it is the model
Yehia was fine-tuned from (same Llama architecture and a byte-identical tokenizer, Apache-2.0, not gated). That is a
documented deviation: E1 is skipped (the released adapter was trained on Yehia's weights), and E2/E3 then measure ALLaM, not Yehia.

**If the session disconnects:** run cells 1 to 3 again (they rebuild `/content`), then re-run the cell of the step you were in.
Generation resumes from the JSONL on Drive; training resumes from the newest complete checkpoint on Drive (keep `RUN_NAME` unchanged).

## 1. Configuration, Google Drive, secrets

In [ ]:
import os, getpass, pathlib

# ---- edit these -------------------------------------------------------------------------------------------
REPO_URL   = "https://github.com/thebazshah/Shaer.git"
BRANCH     = "dev"
# Base model - exactly one of the two supported choices:
#   "Navid-AI/Yehia-7B-preview"            the paper's model (gated: needs the authors' approval)  -> PAPER FLOW
#   "ALLaM-AI/ALLaM-7B-Instruct-preview"   Yehia's public parent (Apache-2.0, not gated)            -> SUBSTITUTE FLOW
BASE_MODEL = "Navid-AI/Yehia-7B-preview"
FORCE_E1 = False       # substitute flow only: run the released adapter on ALLaM anyway (exploratory, not the paper's E1)
RUN_NAME   = "paper_repro_a100"                      # keep identical across restarts => automatic resume
NUM_SAMPLES = 1        # poems per test prompt in E1/E2/E3. The paper reports one; 3-5 lets you also measure best-of-k.
GEN_BATCH_SIZE = 32    # lower to 16 if you hit out-of-memory during generation
# -----------------------------------------------------------------------------------------------------------

if BASE_MODEL == "Navid-AI/Yehia-7B-preview":
    # PAPER FLOW: the setup of the Shaer paper. E0, E1 (released adapter), E2 (untuned Yehia) and E3 (re-training) all apply,
    # and the results are compared with the paper's Yehia-based numbers.
    PAPER_FLOW, TAG, RUN_E1 = True, "yehia", True
elif BASE_MODEL in ("ALLaM-AI/ALLaM-7B-Instruct-preview", "humain-ai/ALLaM-7B-Instruct-preview"):
    # SUBSTITUTE FLOW: ALLaM is the model Yehia was fine-tuned from (same architecture, byte-identical tokenizer).
    # The released adapter was trained on Yehia's weights, so E1 is skipped; E2/E3 measure ALLaM, not Yehia.
    PAPER_FLOW, TAG, RUN_E1 = False, "allam", bool(FORCE_E1)
else:
    raise ValueError(f"unsupported BASE_MODEL {BASE_MODEL!r}: use 'Navid-AI/Yehia-7B-preview' or 'ALLaM-AI/ALLaM-7B-Instruct-preview'")
DRIVE_ROOT = f"/content/drive/MyDrive/shaer-repro-{TAG}"   # everything is saved here (one folder per base model)

from google.colab import drive, userdata
drive.mount("/content/drive")

def secret(name, prompt=None):
    try:
        v = userdata.get(name)
        if v: return v
    except Exception:
        pass
    return getpass.getpass(prompt) if prompt else ""

os.environ["HF_TOKEN"] = secret("HF_TOKEN", "Hugging Face READ token (hidden): ")
os.environ["GITHUB_TOKEN"] = secret("GITHUB_TOKEN")      # only needed if the fork is private (optional secret)
os.environ.update(REPO_URL=REPO_URL, BRANCH=BRANCH, DRIVE_ROOT=DRIVE_ROOT, RUN_NAME=RUN_NAME,
                  BASE_MODEL=BASE_MODEL, SFT_BASE_MODEL_ID=BASE_MODEL,   # train_sft.py reads SFT_BASE_MODEL_ID
                  PAPER_FLOW="1" if PAPER_FLOW else "0", RUN_E1="1" if RUN_E1 else "0", FORCE_E1="1" if FORCE_E1 else "0",
                  NUM_SAMPLES=str(NUM_SAMPLES), GEN_BATCH_SIZE=str(GEN_BATCH_SIZE),
                  METER_MODEL_ID="Shaer-AI/4BiLSTM-meter-classification-pytorch",
                  SFT_PUSH_TO_HUB="0",                  # never create or push to a Hugging Face repo
                  PY="/content/venv/bin/python", REPO="/content/Shaer")
for sub in ("runs", "sft_outputs", "logs"):
    pathlib.Path(DRIVE_ROOT, sub).mkdir(parents=True, exist_ok=True)
print("HF token set:", bool(os.environ["HF_TOKEN"]), "| base model:", BASE_MODEL, "| Drive folder:", DRIVE_ROOT)
if PAPER_FLOW:
    print("PAPER FLOW: Yehia-7B-preview. E0, E1, E2, E3 run as in the paper.")
else:
    print("SUBSTITUTE FLOW: ALLaM-7B-Instruct-preview. " + ("E1 is FORCED (exploratory: the adapter was trained on Yehia)." if RUN_E1
          else "E1 is skipped (the released adapter was trained on Yehia)."), "E2/E3 are NOT the paper's Yehia numbers.")

## 2. Get the code from GitHub (branch `dev`) and point its output folders at Drive

In [ ]:
%%bash
set -euo pipefail
if [ -n "${GITHUB_TOKEN:-}" ]; then CLONE_URL="${REPO_URL/https:\/\//https://x-access-token:${GITHUB_TOKEN}@}"; else CLONE_URL="$REPO_URL"; fi
if [ -d "$REPO/.git" ]; then
  git -C "$REPO" fetch -q origin "$BRANCH" && git -C "$REPO" checkout -q "$BRANCH" && git -C "$REPO" pull -q --ff-only origin "$BRANCH"
else
  git clone -q --branch "$BRANCH" "$CLONE_URL" "$REPO"
fi
git -C "$REPO" remote set-url origin "$REPO_URL"        # do not keep a token in .git/config
# training checkpoints + logs (sft/outputs) and all experiment outputs (runs) live on Drive
rm -rf "$REPO/sft/outputs" "$REPO/runs"
ln -s "$DRIVE_ROOT/sft_outputs" "$REPO/sft/outputs"
ln -s "$DRIVE_ROOT/runs"        "$REPO/runs"
git -C "$REPO" log --oneline -1

## 3. Training environment (about 5 minutes) and GPU check

The GPU check refuses to continue training on a GPU this stack cannot use faithfully: no bf16 (T4), or a Blackwell
card (compute capability 12.x), for which the pinned torch 2.3.1 / bitsandbytes 0.43.3 have no kernels.

In [ ]:
%%bash
set -euo pipefail
pip -q install uv
if [ ! -x "$PY" ]; then
  uv venv --python 3.12 /content/venv
  uv pip install --python "$PY" torch==2.3.1 --index-url https://download.pytorch.org/whl/cu121
  uv pip install --python "$PY" transformers==4.43.0 peft==0.12.0 bitsandbytes==0.43.3 accelerate==0.31.0 \
      datasets==2.20.0 huggingface_hub==0.23.4 tokenizers==0.19.1 safetensors==0.4.3 numpy==1.26.4 pyarrow==17.0.0 \
      pandas==2.2.2 scipy==1.13.1 scikit-learn==1.5.1 sentencepiece==0.2.0 protobuf==4.25.3 python-dotenv==1.0.1 pyyaml httpx matplotlib==3.9.2
fi
uv pip check --python "$PY"
nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv
"$PY" - <<'EOF'
import torch, transformers, peft, bitsandbytes
print("torch", torch.__version__, "| transformers", transformers.__version__, "| peft", peft.__version__, "| bitsandbytes", bitsandbytes.__version__)
assert torch.cuda.is_available(), "no GPU: choose Runtime > Change runtime type > A100"
p = torch.cuda.get_device_properties(0); cc = (p.major, p.minor); gb = p.total_memory / 2**30
print(f"GPU: {p.name} | {gb:.0f} GB | compute capability {cc[0]}.{cc[1]} | bf16 supported: {torch.cuda.is_bf16_supported()}")
ok_train = torch.cuda.is_bf16_supported() and cc < (12, 0) and gb >= 38
ok_gen = torch.cuda.is_bf16_supported() and cc < (12, 0) and gb >= 22
print("E0 (scoring)  : fine on any runtime")
print("E1/E2 (generate):", "OK" if ok_gen else "NOT OK on this GPU (needs bf16, < Blackwell, >= 22 GB)")
print("E3 (training)  :", "OK" if ok_train else "NOT RECOMMENDED on this GPU (needs bf16, < Blackwell, >= 40 GB class)")
EOF
"$PY" -m bitsandbytes 2>&1 | tail -5

In [ ]:
# Hugging Face access check (gated base model + public data). Fails early with a clear message.
import subprocess
r = subprocess.run([os.environ["PY"], "-c", """
from huggingface_hub import hf_hub_download
import os
hf_hub_download(os.environ["BASE_MODEL"], "config.json", token=os.environ["HF_TOKEN"])
print("access to", os.environ["BASE_MODEL"], ": OK")
"""], capture_output=True, text=True)
print(r.stdout.strip() or r.stderr.strip()[-600:])

## 4. E0 - re-score the authors' released poems

Uses the repository's own evaluator, unchanged, on the CPU (deterministic). On the 3,481 released poems it reproduces
the paper's table: meter conformity 0.9064, base-meter accuracy 95.17 %, exact count 83.40 %, count adherence 0.9792.

In [ ]:
%%bash
set -euo pipefail
cd "$REPO"
$PY evaluation/export_released_generations.py --out runs/e0/generations.jsonl
cd evaluation
CUDA_VISIBLE_DEVICES="" $PY score_meter_count.py --input-jsonl ../runs/e0/generations.jsonl --output-jsonl ../runs/e0/scored.jsonl
cd ..
$PY evaluation/summarize_structural.py --scored runs/e0/scored.jsonl --out runs/e0/summary.json > /dev/null
echo "E0 done -> $DRIVE_ROOT/runs/e0"

## 5. E1 - released adapter (`Shaer-AI/Shaer-adapters`, `adapters/fresh_sft/train/best`)

Decoding follows the paper's Table 15 (T 0.55, top-p 0.90, repetition penalty 1.08, 768 new tokens). The base model is
loaded in bf16 without quantisation (the adapter was trained on an NF4 base; add `--quantize nf4` to match that instead).

In [ ]:
%%bash
set -euo pipefail
cd "$REPO"
if [ "$PAPER_FLOW" = "1" ]; then
  EXTRA=""                                        # paper flow: released adapter on Yehia-7B-preview
elif [ "$RUN_E1" = "1" ]; then
  EXTRA="--allow-adapter-mismatch"                # substitute flow, FORCE_E1 = True
  echo "EXPLORATORY: the released adapter was trained on Yehia, not on $BASE_MODEL. This is NOT the paper's E1."
else
  echo "E1 skipped: the released adapter was trained on Yehia-7B-preview, so it is not valid on $BASE_MODEL."
  echo "(set FORCE_E1 = True in the first cell for an exploratory run)"; exit 0
fi
$PY -u evaluation/generate_shaer_test.py $EXTRA --base-model "$BASE_MODEL" --out runs/e1/generations.jsonl --num-samples "$NUM_SAMPLES" --batch-size "$GEN_BATCH_SIZE" 2>&1 | grep -v -i warn
(cd evaluation && CUDA_VISIBLE_DEVICES="" $PY score_meter_count.py --input-jsonl ../runs/e1/generations.jsonl --output-jsonl ../runs/e1/scored.jsonl)
$PY evaluation/summarize_structural.py --scored runs/e1/scored.jsonl --out runs/e1/summary.json > /dev/null
echo "E1 done -> $DRIVE_ROOT/runs/e1"

## 6. E2 - untuned Yehia-7B (same prompts, no adapter)

In [ ]:
%%bash
set -euo pipefail
cd "$REPO"
$PY -u evaluation/generate_shaer_test.py --base-model "$BASE_MODEL" --no-adapter --out runs/e2/generations.jsonl --num-samples "$NUM_SAMPLES" --batch-size "$GEN_BATCH_SIZE" 2>&1 | grep -v -i warn
(cd evaluation && CUDA_VISIBLE_DEVICES="" $PY score_meter_count.py --input-jsonl ../runs/e2/generations.jsonl --output-jsonl ../runs/e2/scored.jsonl)
$PY evaluation/summarize_structural.py --scored runs/e2/scored.jsonl --out runs/e2/summary.json > /dev/null
echo "E2 done -> $DRIVE_ROOT/runs/e2"

## 7. E3 - re-train

**7a. Smoke test first (about 10 minutes, includes the 15 GB model download).** It trains 6 steps with the real 4-bit
configuration, which proves bitsandbytes, the data path and the checkpointing work on this runtime before you commit hours to it.

In [ ]:
%%bash
set -euo pipefail
cd "$REPO/sft"
$PY -u train_sft.py --mode sanity --run_name "sanity_${RUN_NAME}" 2>&1 | grep -v -i warn | tail -25

**7b. Full run.** Starts in the background and logs to Drive, so closing the browser tab does not stop it (a
disconnected or recycled runtime does; just re-run cells 1-3 and this cell, it resumes from the newest complete
checkpoint, written every 200 steps: at most ~200 steps are lost, and a half-written checkpoint is skipped automatically). Optional speed-up: set `SFT_GRADIENT_CHECKPOINTING=0` below; it does not change
the maths, and an 80 GB card has the memory for it.

In [ ]:
%%bash
set -euo pipefail
cd "$REPO/sft"
if pgrep -f "train_sft.py --mode train" >/dev/null; then echo "training is already running"; exit 0; fi
if [ -f "$DRIVE_ROOT/sft_outputs/train/$RUN_NAME/run_summary.json" ]; then
  echo "run '$RUN_NAME' already finished (run_summary.json exists): go to 7c. Use a new RUN_NAME to train again."; exit 0
fi
export SFT_RESUME_MODE=auto            # resumes from the newest complete checkpoint of this RUN_NAME on Drive, if any
# export SFT_GRADIENT_CHECKPOINTING=0  # optional: faster, uses more memory
nohup $PY -u train_sft.py --mode train --run_name "$RUN_NAME" >> "$DRIVE_ROOT/logs/train_${RUN_NAME}.log" 2>&1 &
echo "started pid $! ; log: $DRIVE_ROOT/logs/train_${RUN_NAME}.log"

In [ ]:
%%bash
set -euo pipefail
# progress: run this cell whenever you like
pgrep -f "train_sft.py --mode train" >/dev/null && echo "RUNNING" || echo "NOT running"
tail -c 1200 "$DRIVE_ROOT/logs/train_${RUN_NAME}.log" | tr '\r' '\n' | grep -v -i warn | tail -4
ls "$DRIVE_ROOT/sft_outputs/train/$RUN_NAME" 2>/dev/null | grep checkpoint || true

**7c. After training finished:** generate with the best checkpoint (lowest eval loss), score, summarise.

In [ ]:
%%bash
set -euo pipefail
cd "$REPO"
SUMMARY="$DRIVE_ROOT/sft_outputs/train/$RUN_NAME/run_summary.json"
[ -f "$SUMMARY" ] || { echo "training has not finished (no run_summary.json yet)"; exit 1; }
BEST=$($PY -c "import json;print(json.load(open('$SUMMARY'))['best_eval_checkpoint'])")
echo "best checkpoint: $BEST"; [ -d "$BEST" ] || { echo "checkpoint folder is missing"; exit 1; }
$PY -u evaluation/generate_shaer_test.py --base-model "$BASE_MODEL" --adapter-path "$BEST" --out runs/e3/generations.jsonl --num-samples "$NUM_SAMPLES" --batch-size "$GEN_BATCH_SIZE" 2>&1 | grep -v -i warn
(cd evaluation && CUDA_VISIBLE_DEVICES="" $PY score_meter_count.py --input-jsonl ../runs/e3/generations.jsonl --output-jsonl ../runs/e3/scored.jsonl)
$PY evaluation/summarize_structural.py --scored runs/e3/scored.jsonl --out runs/e3/summary.json > /dev/null
echo "E3 done -> $DRIVE_ROOT/runs/e3"

## 8. Results: your runs against the paper (saved to Drive as `results.md`)

In [ ]:
import json, pathlib
R = pathlib.Path(os.environ["DRIVE_ROOT"], "runs")
PAPER_FLOW = os.environ["PAPER_FLOW"] == "1"
suffix = "" if PAPER_FLOW else ", Yehia-based"       # the paper's rows are Yehia numbers even when we run the substitute
PAPER = {f"Shaer (paper{suffix})": (95.17, 0.9064, 83.40, 0.9792), f"Yehia-7B (paper{suffix})": (26.49, 0.1494, 44.47, 0.7680)}
BASE = os.environ["BASE_MODEL"].split("/")[-1]
def row(name, s):
    return f"| {name} | {s['base_meter_accuracy_pct']:.2f} | {s['meter_conformity']:.4f} | {s['exact_count_accuracy_pct']:.2f} | {s['count_adherence']:.4f} |"
lines = ["| System | Base-meter acc. (%) | Meter conformity | Exact count (%) | Count adherence |", "|---|---:|---:|---:|---:|"]
for n, (a, b, c, d) in PAPER.items(): lines.append(f"| {n} | {a:.2f} | {b:.4f} | {c:.2f} | {d:.4f} |")
if PAPER_FLOW:
    steps = [("e0", "E0 released poems, re-scored"), ("e1", "E1 released adapter"), ("e2", "E2 untuned Yehia-7B"), ("e3", "E3 re-trained here")]
else:
    steps = [("e0", "E0 released poems, re-scored"), ("e1", f"E1 EXPLORATORY: Yehia adapter on {BASE}"),
             ("e2", f"E2 untuned {BASE} (substitute baseline)"), ("e3", f"E3 re-trained on {BASE}")]
for key, name in steps:
    f = R / key / "summary.json"
    if not f.exists(): continue
    d = json.loads(f.read_text())
    if "overall" in d: lines.append(row(name, d["overall"]))
    else:
        lines.append(row(name + " (first sample)", d["first_sample"]))
        lines.append(row(name + f" (best of {d['samples_per_prompt']} by meter score)", d["best_of_k_by_meter"]))
table = "\n".join(lines)
if not PAPER_FLOW:
    table += f"\n\nSubstitute base model ({BASE}): E2/E3 are not comparable one-to-one with the paper's Yehia-based numbers."
(R / "results.md").write_text(table + "\n")
print(table)
print("\nBase-meter accuracy here = first-bayt prediction, the definition that reproduces the paper's own tables;")
print("the majority-vote variant described in the paper's text is in each summary.json (base_meter_accuracy_majority_pct).")

In [ ]:
# E3 training curves from the newest complete checkpoint's trainer_state.json (saved to Drive)
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
run = pathlib.Path(os.environ["DRIVE_ROOT"], "sft_outputs", "train", os.environ["RUN_NAME"])
states = sorted(run.glob("checkpoint-*/trainer_state.json"), key=lambda p: int(p.parent.name.split("-")[1]))
if states:
    h = json.loads(states[-1].read_text())["log_history"]
    tr = [(x["step"], x["loss"]) for x in h if "loss" in x]; ev = [(x["step"], x["eval_loss"]) for x in h if "eval_loss" in x]
    fig, ax = plt.subplots(figsize=(7, 3.6))
    ax.plot(*zip(*tr), lw=0.8, label="train loss"); ax.plot(*zip(*ev), "o-", ms=3, label="eval loss")
    ax.set_xlabel("optimiser step"); ax.set_ylabel("cross-entropy"); ax.grid(alpha=0.3); ax.legend()
    out = pathlib.Path(os.environ["DRIVE_ROOT"], "runs", "training_curves.png"); fig.tight_layout(); fig.savefig(out, dpi=150); print("saved", out)
    print("best eval loss so far:", min(e[1] for e in ev), "| paper: best 2.2074 at step 3000, test 2.1932")
else:
    print("no checkpoint yet")